In [1]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f = xW + b $$
$$ \frac{\partial f}{\partial x} = W $$
$$ \frac{\partial f}{\partial W} = x $$
$$ \frac{\partial f}{\partial b} = 1 $$

In [ ]:
import torch.autograd as autograd
import torch.nn as nn
import torch

import import_ipynb
from common import assert_eq # type: ignore


def linear(x: torch.Tensor, W: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    # (samples, output) = (samples, features) @ (features, output) + (output,)
    return torch.addmm(b, x, W)

class LinearFunction(autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, W: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        # `save_for_backward` is accessible thanks to the `Function` base class.
        ctx.save_for_backward(x, W)
        return linear(x, W, b)

    @staticmethod
    def backward(ctx, dF_df: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # `saved_tensors` is accessible thanks to the `Function` base class.
        x, W = ctx.saved_tensors
        samples, features = x.shape
        output = W.shape[1]
       
        # (samples, features) = (samples, output) @ (output, features)
        dF_dx = dF_df @ W.T
        assert_eq(dF_dx.shape, x.shape)
        assert_eq(dF_dx.shape, (samples, features))
        
        # (features, output) = (features, samples) @ (samples, output)
        dF_dW = x.T @ dF_df          
        assert_eq(dF_dW.shape, W.shape)
        assert_eq(dF_dW.shape, (features, output))
        
        # (output,) * (output,) -> (output, )
        dF_db = 1 * dF_df.sum(dim=0)     
        assert_eq(dF_db.shape, (output,))
        
        return (dF_dx, dF_dW, dF_db)
    

# `LinearFunction` implements the actual operator.
# `Linear(Module)` is just a wrapper for nicer API.
class Linear(nn.Module):
    def __init__(self, in_features, out_features, W=None, b=None):
        super().__init__()

        if W is not None:
            assert_eq(W.shape, (in_features, out_features))
            self.W = nn.Parameter(W)
        else:
            self.W = nn.Parameter(torch.randn(in_features, out_features) * 0.01)
        
        if b is not None:
            assert_eq(b.shape, (out_features,))
            self.b = nn.Parameter(b)
        else:
            self.b = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return LinearFunction.apply(x, self.W, self.b)
    
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)


def test_linear() -> None:
    torch.manual_seed(0)

    samples = 10
    in_features = 3
    out_features = 4
    x = torch.randn(samples, in_features, dtype=torch.float32, requires_grad=True)
    W = torch.randn(in_features, out_features, dtype=torch.float32, requires_grad=True)
    b = torch.randn(out_features, dtype=torch.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    W1 = W.clone().detach().requires_grad_(True)
    b1 = b.clone().detach().requires_grad_(True)
    F1 = linear(x1, W1, b1).sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    W2 = W.clone().detach().requires_grad_(True)
    b2 = b.clone().detach().requires_grad_(True)
    F2 = (x2 @ W2 + b2).sum()
    F2.backward()

    assert_eq(x1, x2, atol=0.001)
    assert_eq(x1.grad, x2.grad, atol=0.001)
    assert_eq(W1, W2, atol=0.001)
    assert_eq(W1.grad, W2.grad, atol=0.001)
    assert_eq(b1, b2, atol=0.001)
    assert_eq(b1.grad, b2.grad, atol=0.001)


if __name__ == "__main__":
    test_linear()
